In [ ]:
import os, sys, time, importlib, itertools, math
from collections import defaultdict
from pathlib import Path
import numpy as np
from kaggle_environments import make

In [ ]:
def _field(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def run_single_game(agent1, agent2, seed=None, config=None):
    cfg = {"seed": seed}
    if config:
        cfg.update(config)

    env = make("orbit_wars", configuration=cfg, debug=False)
    env.run([agent1, agent2])

    final_step = env.steps[-1]
    statuses = [_field(s, "status", None) for s in final_step]
    rewards = [_field(s, "reward", 0) for s in final_step]

    errors = {}
    for i, status in enumerate(statuses):
        if status != "DONE":
            errors[i] = status  # "ERROR", "TIMEOUT", "INVALID", ...

    if errors:
        return None, errors

    r0 = 0.0 if rewards[0] is None else float(rewards[0])
    r1 = 0.0 if rewards[1] is None else float(rewards[1])

    if r0 > r1:
        return 0, None
    elif r1 > r0:
        return 1, None
    else:
        return -1, None

In [ ]:
def _stable_seed(name1, name2, game_idx):
    # Deterministic seed independent of Python hash randomization
    a = sum((i + 1) * ord(c) for i, c in enumerate(name1))
    b = sum((i + 1) * ord(c) for i, c in enumerate(name2))
    return int((a * 1000003 + b * 9176 + game_idx * 7919) % (2**31 - 1))


def run_match(agent1, agent2, name1, name2, n_games=50, config=None):
    wins = [0, 0]
    draws = 0
    errors = []

    for game in range(n_games):
        seed = _stable_seed(name1, name2, game)
        winner, err = run_single_game(agent1, agent2, seed=seed, config=config)

        if err:
            errors.append((game, err))
            # Dung som khi gap loi
            return wins, draws, errors

        if winner == 0:
            wins[0] += 1
        elif winner == 1:
            wins[1] += 1
        else:
            draws += 1

    return wins, draws, errors

In [ ]:
AGENT_DIR = Path("RL_agent")
agents = {}  # name -> ham agent

for file in sorted(AGENT_DIR.glob("*.py")):
    if file.name.startswith("_"):
        continue

    try:
        spec = importlib.util.spec_from_file_location(file.stem, str(file))
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
    except Exception as e:
        print(f"WARNING: Loi import {file.name}: {e}")
        continue

    if hasattr(module, "agent") and callable(module.agent):
        agents[file.stem] = module.agent
    else:
        print(f"WARNING: {file} khong co ham agent()")

print(f"Da load {len(agents)} agent: {list(agents.keys())}")

In [ ]:
disqualified = set()  # agent bi loai vi loi
error_log = []        # ghi chi tiet loi
valid_agents = set(agents.keys())

In [ ]:
results = []
total_wins = defaultdict(int)

if len(valid_agents) < 2:
    print("Khong du agent hop le de thi dau vong tron (can >= 2).")
else:
    for name1, name2 in itertools.combinations(sorted(valid_agents), 2):
        if name1 in disqualified or name2 in disqualified:
            continue

        agent1 = agents[name1]
        agent2 = agents[name2]

        print(f"Dang dau: {name1} vs {name2}")
        wins, draws, errors = run_match(agent1, agent2, name1=name1, name2=name2, n_games=50)

        if errors:
            faulty_agents = set()
            for game_idx, err_dict in errors:
                for agent_idx, status in err_dict.items():
                    faulty = name1 if agent_idx == 0 else name2
                    faulty_agents.add(faulty)
                    disqualified.add(faulty)
                    error_log.append(
                        f"Loi {status} o game {game_idx} giua {name1} va {name2}: {faulty}"
                    )
            print(f"  -> Loi: {', '.join(sorted(faulty_agents))} bi loai.")
            continue

        total_wins[name1] += wins[0]
        total_wins[name2] += wins[1]

        score1 = wins[0] / 50.0
        score2 = wins[1] / 50.0

        results.append({
            "match": f"{name1} vs {name2}",
            "score": f"{score1:.2f} vs {score2:.2f}",
            "wins": wins,
            "draws": draws,
        })

        print(f"  Ket qua: {wins[0]} - {wins[1]} (hoa {draws})")

In [ ]:
with open("results.txt", "w", encoding="utf-8") as f:
    f.write("KET QUA GIAI DAU ORBIT WARS")
    f.write("\n")
    f.write("============================")
    f.write("\n\n")

    if results:
        for r in results:
            f.write(f"Van dau({r['match']}), ti so({r['score']}), hoa({r['draws']})")
            f.write("\n")
    else:
        f.write("Chua co tran hop le nao duoc ghi nhan.")
        f.write("\n")

    f.write("\nTONG SO TRAN THANG CUA MOI AGENT:\n")
    for name in sorted(valid_agents):
        if name not in disqualified:
            f.write(f"{name}: {total_wins[name]} thang")
            f.write("\n")

    f.write("\nAGENT BI LOI:\n")
    if error_log:
        for err in error_log:
            f.write(err)
            f.write("\n")
    else:
        f.write("Khong co loi.\n")

print("Da ghi ket qua vao results.txt")

In [ ]:
print()
print("=== KET QUA ===")
if results:
    for r in results:
        print(f"{r['match']}: {r['score']} (hoa {r['draws']})")
else:
    print("Khong co ket qua hop le nao.")

print()
print("Tong thang:")
for name in sorted(valid_agents):
    if name not in disqualified:
        print(f"{name}: {total_wins[name]}")

if disqualified:
    print()
    print("Agent bi loai vi loi:")
    for name in sorted(disqualified):
        print(name)

if error_log:
    print()
    print("Chi tiet loi:")
    for err in error_log:
        print("-", err)